# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier if hasattr(metadata, 'identifier') else 'N/A'}")
print(f"License: {metadata.license if hasattr(metadata, 'license') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset with their @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rec in record_sets:
    print(f"- @id: {rec['@id']}  |  Name: {rec.get('name', 'N/A')}")

# For illustrative purposes, pick the first record set if available
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set (@id: {record_set_id}):")
    # List the fields & columns of this record set
    fields = record_sets[0].get('field', [])
    if fields:
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"  - Field @id: {field_id}")
    else:
        print("  No fields found.")
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s.

In [ ]:
# Extract all record sets by @id, load into DataFrames
dataframes = {}
record_set_ids = [r['@id'] for r in record_sets]

for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rid] = df
            print(f"Loaded DataFrame for record set {rid} with {df.shape[0]} rows and {df.shape[1]} columns.")
        else:
            print(f"No records found for record set {rid}.")
    except Exception as e:
        print(f"Could not load records for {rid}: {e}")

# Display the first DataFrame's column names and a preview
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first record set ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No DataFrames to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. Filter records by a numeric field, normalize values, and group by a categorical field where possible.

In [ ]:
# For EDA: select a record set and fields for numeric/categorical analysis
if dataframes:
    df = dataframes[first_rs_id]
    print(f"First 5 rows of DataFrame ({first_rs_id}):")
    display(df.head())

    # Find the first numeric column for demonstration
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        try:
            # Try to convert possible object columns
            for col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='ignore')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
        except Exception:
            pass

    if numeric_field:
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].dtype.kind in 'fi' else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric field found for EDA demonstration.")

    # Attempt to group by a categorical field (non-numeric, non-id)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if numeric_field and group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} and mean of {numeric_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping was possible
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its Croissant schema and the `mlcroissant` library. We:
- Examined available record sets and their fields using their `@id`s,
- Loaded and explored the first record set as a DataFrame,
- Performed basic filtering, normalization, and grouping operations for EDA,
- Generated example visualizations (distribution, grouped means).

Further analyses can build on this foundation: for example, correlating variables, addressing data biases, or performing regression modeling using the adoption predictors. The Croissant metadata and `@id` referencing facilitates reproducibility and consistency.

**For more, see the [mlcroissant documentation](https://github.com/mlcommons/croissant).**